# Logger session: head orientation, pose tracking, robot interaction, spikes
One wireless-logger session (rat on a round table with a walking robot, 6 cameras, IMU on the head-mounted logger, Intan512 ephys).
Everything here is loaded from analytics; the notebook only plots and runs the tuning analyses.

| analytic | what | computed with |
|---|---|---|
| `HeadOrientation` | IMU head angles, 100 Hz, drift-corrected, anatomical names | `python run_pipeline.py HeadOrientation --session_names <s>` (any env) |
| `BehaviorPose` | raw DLC keypoints on cam2, 30 Hz | DEEPLABCUT env, GPU: `python run_pipeline.py BehaviorPose --session_names <s>` |
| `ArenaPoseFeatures` | smoothed keypoints, rat + robot position, IMU angles per frame, robot bearing, turns | `python run_pipeline.py ArenaPoseFeatures --session_names <s>` (needs the two above) |
| `Spikes`, `SpikeClusterMetadata` | curated JRClust spikes with reliability flags | `python run_pipeline.py Spikes-SpikeClusterMetadata --session_names <s>` |

Sections: 1 what is what · 2 head orientation · 3 how certain the pose tracking is · 4 behaviour and attention to the robot ·
5 the session video · 6 spikes · 7 head-direction tuning · 8 snout-elevation tuning

In [ ]:
import os
import sys

# to setup import paths add project root dirs to sys.path
sys.path.append(os.path.join(os.getcwd(), "..", ".."))
from baseVR.base_functionality import init_import_paths
init_import_paths()

In [2]:
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cv2

from analytics_processing import analytics
import analytics_processing.analytics_constants as C
import analytics_processing.sessions_from_nas_parsing as sp
from analytics_processing.modality_loading import session_modality_from_nas
from CustomLogger import CustomLogger as Logger

Logger().init_logger(None, None, logging_level="WARNING")
plt.rcParams.update({'figure.dpi': 110, 'axes.spines.top': False, 'axes.spines.right': False, 'font.size': 9})

In [3]:
SESSION = '2026-09-09_15-15_rPG007_P1500_RobotInteract_45min'
F = sp.sessionnames2fullfnames([SESSION])[0][0]

data = {}
for analytic in ['HeadOrientation', 'ArenaPoseFeatures', 'Spikes', 'SpikeClusterMetadata']:
    data[analytic] = analytics.get_analytics(analytic, session_names=[SESSION]).reset_index(drop=True)
problems = session_modality_from_nas(F, 'problem_intervals')

head, pose = data['HeadOrientation'], data['ArenaPoseFeatures']
spikes, units = data['Spikes'], data['SpikeClusterMetadata']
t_head = head.ephys_timestamp.values / 1e6          # s, ephys clock (all tables share it)
t_pose = pose.image_ephys_timestamp.values / 1e6
malf = problems[problems.category == 'logger_malfunction_window']
MAL = (malf.start_sample.iloc[0] / 20_000, malf.end_sample.iloc[0] / 20_000)

def shade(ax):
    """logger malfunction window"""
    ax.axvspan(*MAL, color='crimson', alpha=.15, lw=0)

for k, v in data.items():
    print(f'{k:22s} {v.shape}')
print('pose model:', pose.dlc_scorer.iloc[0])

HeadOrientation        (275263, 13)
ArenaPoseFeatures      (81000, 81)
Spikes                 (4386219, 19)
SpikeClusterMetadata   (225, 26)
pose model: DLC_Resnet50_ratArenaLoggerOct28shuffle6_snapshot_1300


In [ ]:
analytics.get_analytics('SpikeClusterMetadata', session_names=[SESSION]).reset_index(drop=True)


,sample_id,cluster_id_ssbatch,jrc_site,amplitude_uV,ephys_timestamp,channel,channel_2nd,channel_3rd,shank,ss_batch_id,cluster_id,depth,cluster_id_str,cluster_color,gross_brain_area,fine_brain_area,in_zero_pad,within_5ms_of_pad,in_malfunction_window
0,4,13,108,-172.575,200,440,442,NaN,2.0,0,13,1871.2073,Unit0013,#B2828D,shank2,shank2,False,False,False
1,35,86,202,-100.425,1750,317,300,NaN,4.0,0,86,5343.6808,Unit0086,#6C7C32,shank4,shank4,False,False,False
2,36,18,110,-132.600,1800,442,423,NaN,2.0,0,18,1681.4641,Unit0018,#620042,shank2,shank2,False,False,False
3,62,513,464,-120.900,3100,47,51,NaN,8.0,0,204,4492.3236,Unit0204,#FC0080,shank8,shank8,False,False,False
4,93,24,112,-248.040,4650,444,421,NaN,2.0,0,24,1492.2906,Unit0024,#2E91E5,shank2,shank2,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4386214,55052290,30,118,-156.780,2752614500,445,447,NaN,2.0,0,30,928.2329,Unit0030,#B68100,shank2,shank2,False,False,False
4386215,55052330,17,109,-108.030,2752616500,423,421,NaN,2.0,0,17,1779.2073,Unit0017,#A777F1,shank2,shank2,False,False,False
4386216,55052415,30,118,-126.945,2752620750,445,423,NaN,2.0,0,30,928.2329,Unit0030,#B68100,shank2,shank2,False,False,False
4386217,55052451,369,376,-124.020,2752622550,194,189,NaN,6.0,0,178,395.3849,Unit0178,#00A08B,shank6,shank6,False,False,False
